In [1]:
## SQL Databases

In [2]:
## create sample SQLite Database
import sqlite3
import os

os.makedirs("data/databases", exist_ok = True)

In [4]:
## create a sample database
conn = sqlite3.connect('data/databases/company.db')
cursor = conn.cursor()

In [5]:
# create tables
cursor.execute (''' CREATE TABLE IF NOT EXISTS employees
                (id INTEGER PRIMARY KEY, name TEXT, role TEXT, department TEXT, salary REAL)''')

In [6]:
cursor.execute(''' CREATE TABLE IF NOT EXISTS projects
                (id INTEGER PRIMARY KEY, name TEXT, status TEXT, budget REAL, lead_id INTEGER)''')

In [7]:
# Insert sample data
employees = [
    (1, 'John Doe', 'Senior Developer', 'Engineering', 95000),
    (2, 'Jane Smith', 'Data Scientist', 'Analytics', 105000),
    (3, 'Mike Johnson', 'Product Manager', 'Product', 110000),
    (4, 'Sarah Williams', 'DevOps Engineer', 'Engineering', 98000)
]

projects = [
    (1, 'RAG Implementation', 'Active', 150000, 1),
    (2, 'Data Pipeline', 'Completed', 80000, 2),
    (3, 'Customer Portal', 'Planning', 200000, 3),
    (4, 'ML Platform', 'Active', 250000, 2)
]


In [8]:
cursor.executemany('INSERT OR REPLACE INTO employees VALUES (?,?,?,?,?)', employees)
cursor.executemany('INSERT OR REPLACE INTO projects VALUES (?,?,?,?,?)', projects)

In [9]:
cursor.execute("SELECT * from employees")

In [10]:
conn.commit()
conn.close()

In [11]:
## Database content Extraction

In [13]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

d:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# Method 1: SQL Database Utility
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

## get database info
print(f"Tables : {db.get_usable_table_names()}")
print(f"\nTable DDL:")
print(db.get_table_info())

Tables : ['employees', 'projects']

Table DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	John Doe	Senior Developer	Engineering	95000.0
2	Jane Smith	Data Scientist	Analytics	105000.0
3	Mike Johnson	Product Manager	Product	110000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	RAG Implementation	Active	150000.0	1
2	Data Pipeline	Completed	80000.0	2
3	Customer Portal	Planning	200000.0	3
*/


In [ ]:
from typing import List
from langchain_core.documents import Document
# Method 2:  Custom SQL to Dcoument conversion
print("\n Custom SQL Processing")

def sql_to_documents(db_path: str) -> List[Document]:
    """ Convert SQL database to documents with conext"""
    conn=sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents = []
    # Strategy  1: Create documents for each table
    cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]

        # Get table schema
        cursor.execute(f"PRAGMA table_info({table_name})")
    return documents